# MobileNetV3-Small: новый supervised baseline

Основные метрики: Macro-F1 и Accuracy до отказа; FP по классам и External FPR после отказа. Эпоха выбирается только по val Macro-F1, затем Accuracy. Порог отказа определяется только по negative_val. Данные — проверенные manual и Wikimedia; исходники не изменяются. Старый test является диагностическим, а не новым закрытым полевым тестом.


In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/small_classifier.py").exists())
sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "models/mobilenetv3_small_classifier"
CONFIG = json.loads((ROOT / "config/mobilenetv3_small_classifier.json").read_text(encoding="utf-8"))
WEIGHTS = ROOT / ".runtime/torch/checkpoints/mobilenet_v3_small-047dcff4.pth"


## Обучение

Для повторного независимого эксперимента укажите новый OUTPUT. Текущий запуск выполняется через CLI; не запускайте его параллельно в ноутбуке. Веса ImageNet должны быть доступны локально. Неизвестные изменения изображений блокируют запуск до проверки.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from src.small_classifier import run
    run(ROOT, OUTPUT, WEIGHTS, CONFIG)
else:
    print("Обучение выключено; ниже отображаются сохранённые результаты.")


## Данные, выбор эпохи и метрики


In [ ]:
for name in ["data_audit.json", "selection.json", "rejection_policy.json", "metrics.json"]:
    path = OUTPUT / name
    print("\n", name)
    print(path.read_text(encoding="utf-8") if path.exists() else "Ещё не сформирован")


## Диагностика по POI

В test_per_poi.csv FP — число чужих положительных изображений, ошибочно отнесённых к данному POI. В negative_test_predictions.csv accepted=True означает ложное принятие внешнего негатива. Max-softmax используется как score, а не как откалиброванная вероятность.


In [ ]:
import csv
path = OUTPUT / "test_per_poi.csv"
if path.exists():
    with path.open(encoding="utf-8-sig", newline="") as stream:
        per_poi = list(csv.DictReader(stream))
    for row in sorted(per_poi, key=lambda r: float(r["f1"])):
        print(row["poi_id"], "F1:", round(float(row["f1"]), 3), "FP:", row["fp"], "support:", row["support"])


## Ограничения

Не увеличивайте Accuracy за счёт удаления трудных test-фотографий. Все исключения данных фиксируются в data_exclusions.csv, перемещения с исправлением POI — в data_changes.csv. После просмотра ошибок соберите новый независимый test. Снимки negatives с целевым POI на фоне требуют перепроверки разметки.
